# Phase 2 — Mule Account Detection Pipeline

**Scale:** 160K accounts, 400M transactions, 16 GB parquet

| Tier | Patterns |
|---|---|
| **Strong** | H9, R12+H13, H10, H27, H34, H42 |
| **Moderate** | R2, R9, R3, R4+H8+H16, H21, R8, R10, H37, H35, H38, H43, H47 |
| **New (P2)** | Geo-velocity, IP sharing, balance dynamics, scheme risk, demographics |

---

## 0 — Setup & Static Data Loading

In [1]:
import pandas as pd
import numpy as np
from glob import glob
from scipy import stats
from collections import Counter, defaultdict
import warnings, time, os, gc

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)


t0 = time.time()
print("Loading static tables (parquet)...")

customers = pd.read_parquet("customers.parquet")
accounts  = pd.read_parquet("accounts.parquet")
linkage   = pd.read_parquet("customer_account_linkage.parquet")
products  = pd.read_parquet("product_details.parquet")
labels    = pd.read_parquet("train_labels.parquet")
test      = pd.read_parquet("test_accounts.parquet")

# New Phase 2 tables
demographics = pd.read_parquet("demographics.parquet")
acct_extra   = pd.read_parquet("accounts-additional.parquet")
branch_info  = pd.read_parquet("branch.parquet")

# Parse dates
for col in ["account_opening_date", "freeze_date", "unfreeze_date",
            "last_mobile_update_date", "last_kyc_date"]:
    if col in accounts.columns:
        accounts[col] = pd.to_datetime(accounts[col], errors="coerce")
for col in ["date_of_birth", "relationship_start_date"]:
    if col in customers.columns:
        customers[col] = pd.to_datetime(customers[col], errors="coerce")
for col in ["address_last_update_date", "passbook_last_update_date"]:
    if col in demographics.columns:
        demographics[col] = pd.to_datetime(demographics[col], errors="coerce")

all_ids = pd.concat([labels[["account_id"]], test[["account_id"]]]).drop_duplicates()
print(f"Static tables loaded in {time.time()-t0:.1f}s")
print(f"Universe: {len(all_ids):,} accounts (train={len(labels):,}, test={len(test):,})")
print(f"Transactions: {len(glob('/transactions/batch-*/part_*.parquet'))} part files")
print(f"Txn Additional: {len(glob('/transactions_additional/batch-*/part_*.parquet'))} part files")

Loading static tables (parquet)...
Static tables loaded in 0.8s
Universe: 160,153 accounts (train=96,091, test=64,062)
Transactions: 0 part files
Txn Additional: 0 part files


## 1 — Chunked Transaction Feature Engine

Strategy: process transactions **batch-by-batch**, accumulate per-account
statistics in dictionaries, then convert to DataFrame at the end.
This keeps memory usage bounded.

In [ ]:
# Initialize accumulators for all features computed from transactions
from dataclasses import dataclass, field

class AccountAccumulator:
    """Accumulates per-account statistics across chunks."""
    def __init__(self):
        # Basic counts
        self.txn_count = defaultdict(int)
        self.near_threshold_count = defaultdict(int)
        self.round_count = defaultdict(int)
        self.total_volume = defaultdict(float)
        self.total_credit = defaultdict(float)
        self.total_debit = defaultdict(float)
        self.small_txn_count = defaultdict(int)  # H21

        # Amount stats — for online mean/var (Welford)
        self.amt_n = defaultdict(int)
        self.amt_mean = defaultdict(float)
        self.amt_m2 = defaultdict(float)
        self.amt_max = defaultdict(float)

        # Timing — collect gap_hours per account (up to cap)
        self.last_ts = {}      # last timestamp seen per account
        self.gap_hours = defaultdict(list)  # capped list

        # Night transactions (H1 removed, but gap_cv needs timestamps)
        self.night_count = defaultdict(int)

        # Counterparty sets
        self.cp_counter = defaultdict(Counter)  # H16, H34
        self.credit_cps = defaultdict(set)      # R4
        self.debit_cps = defaultdict(set)       # R4

        # Trigram building — action sequences
        self.action_buf = defaultdict(list)     # last 2 actions for trigram sliding

        # MCC tracking
        self.mcc_counter = defaultdict(Counter)  # H10

        # First large txn timestamp — H47
        self.first_large_ts = {}

        # Large credit/debit pairs for dwell time — R3
        self.pending_credit = {}   # account_id -> (ts, amount) of last large credit
        self.dwell_hours_list = defaultdict(list)

        # Daily volume per account (date -> volume) — for phase shift, ROC
        self.daily_vol = defaultdict(lambda: defaultdict(float))

        # Temporal echo — bucket sizes
        self.max_bucket_size = defaultdict(int)

    def process_chunk(self, chunk_df, account_open_dates=None):
        """Process one chunk of transactions."""
        chunk = chunk_df.sort_values("transaction_timestamp")

        rounds = {1000, 5000, 10000, 50000, 100000}

        for row in chunk.itertuples(index=False):
            aid = row.account_id
            amt = abs(row.amount)
            ts = row.transaction_timestamp
            txn_type = row.txn_type
            channel = str(row.channel)
            cp = row.counterparty_id
            mcc = row.mcc_code

            # Basic counts
            self.txn_count[aid] += 1
            self.total_volume[aid] += amt

            if txn_type == "C":
                self.total_credit[aid] += amt
            else:
                self.total_debit[aid] += amt

            # R2 — structuring
            if 45000 <= amt < 50000:
                self.near_threshold_count[aid] += 1

            # R9 — round amounts
            if amt in rounds:
                self.round_count[aid] += 1

            # H21 — small txns
            if amt < 500:
                self.small_txn_count[aid] += 1

            # Online mean/var (Welford)
            self.amt_n[aid] += 1
            delta = amt - self.amt_mean[aid]
            self.amt_mean[aid] += delta / self.amt_n[aid]
            delta2 = amt - self.amt_mean[aid]
            self.amt_m2[aid] += delta * delta2
            self.amt_max[aid] = max(self.amt_max.get(aid, 0), amt)

            # MCC tracking
            self.mcc_counter[aid][mcc] += 1

            # Counterparty
            if pd.notna(cp):
                self.cp_counter[aid][cp] += 1
                if txn_type == "C":
                    self.credit_cps[aid].add(cp)
                else:
                    self.debit_cps[aid].add(cp)

            # Gaps (H35)
            if aid in self.last_ts and pd.notna(ts) and pd.notna(self.last_ts[aid]):
                gap_h = (ts - self.last_ts[aid]).total_seconds() / 3600
                if len(self.gap_hours[aid]) < 5000:  # cap to avoid memory blow
                    self.gap_hours[aid].append(gap_h)
            self.last_ts[aid] = ts

            # Night (00-05)
            if pd.notna(ts) and ts.hour < 6:
                self.night_count[aid] += 1

            # H47 — first large txn
            if amt > 50000 and aid not in self.first_large_ts:
                self.first_large_ts[aid] = ts

            # R3 — dwell time
            if amt >= 50000:
                if txn_type == "C":
                    self.pending_credit[aid] = ts
                elif txn_type == "D" and aid in self.pending_credit:
                    dwell = (ts - self.pending_credit[aid]).total_seconds() / 3600
                    if dwell >= 0:
                        self.dwell_hours_list[aid].append(dwell)
                    del self.pending_credit[aid]

            # Daily volume
            if pd.notna(ts):
                day = ts.date()
                self.daily_vol[aid][day] += amt

            # Trigram buffer
            action = f"{txn_type}_{channel}"
            buf = self.action_buf[aid]
            buf.append(action)

    def process_chunk_fast(self, chunk_df):
        """Vectorized processing of a chunk (much faster than itertuples)."""
        chunk = chunk_df.copy()
        chunk["abs_amount"] = chunk["amount"].abs()
        chunk["date"] = chunk["transaction_timestamp"].dt.date
        chunk["hour"] = chunk["transaction_timestamp"].dt.hour

        # Group by account_id
        grp = chunk.groupby("account_id")

        # Basic counts
        counts = grp.size()
        for aid, cnt in counts.items():
            self.txn_count[aid] += cnt

        # Volume
        vol = grp["abs_amount"].sum()
        for aid, v in vol.items():
            self.total_volume[aid] += v

        # Credit/Debit volume
        credit_mask = chunk["txn_type"] == "C"
        debit_mask = chunk["txn_type"] == "D"
        for aid, v in chunk[credit_mask].groupby("account_id")["abs_amount"].sum().items():
            self.total_credit[aid] += v
        for aid, v in chunk[debit_mask].groupby("account_id")["abs_amount"].sum().items():
            self.total_debit[aid] += v

        # R2 — near threshold
        near = chunk[chunk["abs_amount"].between(45000, 49999)].groupby("account_id").size()
        for aid, cnt in near.items():
            self.near_threshold_count[aid] += cnt

        # R9 — round amounts
        rounds_set = {1000, 5000, 10000, 50000, 100000}
        round_mask = chunk["abs_amount"].isin(rounds_set)
        for aid, cnt in chunk[round_mask].groupby("account_id").size().items():
            self.round_count[aid] += cnt

        # H21 — small txns
        small = chunk[chunk["abs_amount"] < 500].groupby("account_id").size()
        for aid, cnt in small.items():
            self.small_txn_count[aid] += cnt

        # Night txns
        night = chunk[chunk["hour"] < 6].groupby("account_id").size()
        for aid, cnt in night.items():
            self.night_count[aid] += cnt

        # Amount max
        amax = grp["abs_amount"].max()
        for aid, mx in amax.items():
            self.amt_max[aid] = max(self.amt_max.get(aid, 0), mx)

        # MCC counters
        for (aid, mcc), cnt in chunk.groupby(["account_id", "mcc_code"]).size().items():
            self.mcc_counter[aid][mcc] += cnt

        # Counterparty counters
        cp_valid = chunk[chunk["counterparty_id"].notna()]
        for (aid, cp), cnt in cp_valid.groupby(["account_id", "counterparty_id"]).size().items():
            self.cp_counter[aid][cp] += cnt
        for aid, cp in zip(cp_valid[credit_mask[cp_valid.index]]["account_id"],
                           cp_valid[credit_mask[cp_valid.index]]["counterparty_id"]):
            self.credit_cps[aid].add(cp)
        for aid, cp in zip(cp_valid[debit_mask[cp_valid.index]]["account_id"],
                           cp_valid[debit_mask[cp_valid.index]]["counterparty_id"]):
            self.debit_cps[aid].add(cp)

        # Gaps — need sorted per account
        sorted_chunk = chunk.sort_values(["account_id", "transaction_timestamp"])
        sorted_chunk["prev_ts"] = sorted_chunk.groupby("account_id")["transaction_timestamp"].shift(1)
        sorted_chunk["gap_h"] = (sorted_chunk["transaction_timestamp"] - sorted_chunk["prev_ts"]).dt.total_seconds() / 3600
        gaps_valid = sorted_chunk[sorted_chunk["gap_h"].notna()]
        for aid, g in gaps_valid.groupby("account_id")["gap_h"]:
            existing = self.gap_hours[aid]
            remaining = 5000 - len(existing)
            if remaining > 0:
                existing.extend(g.tolist()[:remaining])

        # H47 — first large txn
        large = chunk[chunk["abs_amount"] > 50000]
        if len(large) > 0:
            first_per_acct = large.sort_values("transaction_timestamp").groupby("account_id")["transaction_timestamp"].first()
            for aid, ts in first_per_acct.items():
                if aid not in self.first_large_ts:
                    self.first_large_ts[aid] = ts

        # Daily volume
        daily = chunk.groupby(["account_id", "date"])["abs_amount"].sum()
        for (aid, day), v in daily.items():
            self.daily_vol[aid][day] += v

        # Trigram buffer — append actions in order
        sorted_chunk["action"] = sorted_chunk["txn_type"].astype(str) + "_" + sorted_chunk["channel"].astype(str)
        for aid, g in sorted_chunk.groupby("account_id")["action"]:
            self.action_buf[aid].extend(g.tolist())

    def finalize(self, account_open_dates, mule_trigrams_top):
        """Convert accumulators into an account-level DataFrame."""
        all_aids = set(self.txn_count.keys())
        rows = []

        for aid in all_aids:
            n = self.txn_count[aid]
            row = {"account_id": aid}

            # Structuring (R2, R9)
            row["near_threshold_pct"] = self.near_threshold_count[aid] / max(n, 1)
            row["near_threshold_count"] = self.near_threshold_count[aid]
            row["round_amount_pct"] = self.round_count[aid] / max(n, 1)
            row["txn_count"] = n
            row["total_volume"] = self.total_volume[aid]

            # Amount stats
            row["amt_mean"] = self.amt_mean[aid]
            row["amt_std"] = (self.amt_m2[aid] / max(self.amt_n[aid] - 1, 1)) ** 0.5 if self.amt_n[aid] > 1 else 0
            row["amt_max"] = self.amt_max.get(aid, 0)

            # Timing (H35)
            gaps = self.gap_hours[aid]
            if len(gaps) > 2:
                gap_arr = np.array(gaps)
                g_mean = gap_arr.mean()
                g_std = gap_arr.std()
                row["gap_cv"] = g_std / max(g_mean, 0.01)
                row["gap_mean"] = g_mean
                row["gap_std"] = g_std
            else:
                row["gap_cv"] = np.nan
                row["gap_mean"] = np.nan
                row["gap_std"] = np.nan

            # Night pct
            row["night_txn_pct"] = self.night_count[aid] / max(n, 1)

            # Graph (H34, H16, R4)
            cp_counts = self.cp_counter[aid]
            row["degree_centrality"] = len(cp_counts)
            if len(cp_counts) >= 2:
                total_cp = sum(cp_counts.values())
                probs = np.array(list(cp_counts.values())) / total_cp
                row["counterparty_entropy"] = -np.sum(probs * np.log2(probs))
            else:
                row["counterparty_entropy"] = 0.0
            n_credit_cp = len(self.credit_cps[aid])
            n_debit_cp = max(len(self.debit_cps[aid]), 1)
            row["fanin_fanout_ratio"] = n_credit_cp / n_debit_cp

            # H21 — Life Ratio
            row["life_ratio"] = self.small_txn_count[aid] / max(n, 1)

            # R3 — Dwell time
            dwell_list = self.dwell_hours_list[aid]
            row["median_dwell_hours"] = np.median(dwell_list) if dwell_list else np.nan

            # H38 — Residual ratio (will be merged from accounts later)

            # H47 — Days to first large
            if aid in self.first_large_ts and aid in account_open_dates:
                open_dt = account_open_dates.get(aid)
                if pd.notna(open_dt) and pd.notna(self.first_large_ts[aid]):
                    row["days_to_first_large"] = (self.first_large_ts[aid] - open_dt).days
                else:
                    row["days_to_first_large"] = np.nan
            else:
                row["days_to_first_large"] = np.nan

            # H43 — Amount entropy (from MCC counter as proxy; real entropy needs amounts)
            # We compute the entropy of the MCC distribution instead
            mcc_total = sum(self.mcc_counter[aid].values())
            if mcc_total >= 5:
                probs = np.array(list(self.mcc_counter[aid].values())) / mcc_total
                row["mcc_entropy"] = -np.sum(probs * np.log2(probs))
            else:
                row["mcc_entropy"] = np.nan

            # H27 — Trigrams
            actions = self.action_buf[aid]
            if len(actions) >= 3:
                trigrams = [f"{actions[i]}|{actions[i+1]}|{actions[i+2]}" for i in range(len(actions)-2)]
                row["trigram_diversity"] = len(set(trigrams)) / len(trigrams)
                row["mule_trigram_count"] = sum(1 for t in trigrams if t in mule_trigrams_top)
            else:
                row["trigram_diversity"] = 0.0
                row["mule_trigram_count"] = 0

            # Daily vol stats — for phase shift, velocity
            dv = self.daily_vol[aid]
            if len(dv) >= 7:
                vols = np.array(list(dv.values()))
                vols_sorted = np.sort(vols)
                mid = len(vols_sorted) // 2
                if vols_sorted[:mid].std() > 0 and vols_sorted[mid:].std() > 0:
                    ks_stat, _ = stats.ks_2samp(vols_sorted[:mid], vols_sorted[mid:])
                    row["ks_phase_shift"] = ks_stat
                else:
                    row["ks_phase_shift"] = np.nan
            else:
                row["ks_phase_shift"] = np.nan
            row["active_days"] = len(dv)

            # Temporal echo (will be computed separately in batch)
            row["max_temporal_cohort"] = self.max_bucket_size.get(aid, 0)

            rows.append(row)

        return pd.DataFrame(rows)

print("✓ AccountAccumulator defined")

## 2 — Discover Mule-Enriched Trigrams (Train Only)

In [ ]:
print("Discovering mule-enriched trigrams from training data (sampling)...")
t_tri = time.time()

# Read a subset of transaction parts to build trigram enrichment
mule_ids_set = set(labels[labels["is_mule"] == 1]["account_id"].head(500))
legit_ids_set = set(labels[labels["is_mule"] == 0]["account_id"].head(500))
sample_ids = mule_ids_set | legit_ids_set

mule_lookup = dict(zip(labels["account_id"], labels["is_mule"]))

trigrams_by_class = {0: Counter(), 1: Counter()}
parts = sorted(glob(f"{DATA_DIR}/transactions/batch-*/part_*.parquet"))

for i, part_path in enumerate(parts[:50]):  # sample first 50 parts
    df = pd.read_parquet(part_path, columns=["account_id", "transaction_timestamp", "txn_type", "channel"])
    df = df[df["account_id"].isin(sample_ids)]
    if len(df) == 0:
        continue
    df = df.sort_values("transaction_timestamp")
    df["action"] = df["txn_type"].astype(str) + "_" + df["channel"].astype(str)
    for aid, grp in df.groupby("account_id"):
        is_m = mule_lookup.get(aid, -1)
        if is_m < 0:
            continue
        actions = grp["action"].tolist()
        for j in range(len(actions) - 2):
            trigrams_by_class[is_m][f"{actions[j]}|{actions[j+1]}|{actions[j+2]}"] += 1
    del df; gc.collect()

total_l = sum(trigrams_by_class[0].values()) or 1
total_m = sum(trigrams_by_class[1].values()) or 1

enrichment = []
for tri in set(trigrams_by_class[0]) | set(trigrams_by_class[1]):
    mc = trigrams_by_class[1].get(tri, 0)
    if mc >= 5:
        enrichment.append((tri, (mc / total_m) / max(trigrams_by_class[0].get(tri, 0) / total_l, 1e-8)))
enrichment.sort(key=lambda x: -x[1])
MULE_TRIGRAMS_TOP10 = set(t for t, _ in enrichment[:10])

print(f"Top-10 mule trigrams ({time.time()-t_tri:.0f}s):")
for t, r in enrichment[:10]:
    print(f"  {t:<55} {r:>8.0f}x")

## 3 — Process All Transactions (Chunked)

In [ ]:
acc = AccountAccumulator()
account_open = dict(zip(accounts["account_id"], accounts["account_opening_date"]))

parts = sorted(glob(f"{DATA_DIR}/transactions/batch-*/part_*.parquet"))
print(f"Processing {len(parts)} transaction parts...")
t_proc = time.time()

for i, part_path in enumerate(parts):
    df = pd.read_parquet(part_path)
    df["transaction_timestamp"] = pd.to_datetime(df["transaction_timestamp"], errors="coerce")
    acc.process_chunk_fast(df)

    if (i + 1) % 50 == 0 or i == len(parts) - 1:
        elapsed = time.time() - t_proc
        rate = (i + 1) / elapsed * 60
        print(f"  Part {i+1}/{len(parts)} — {rate:.0f} parts/min — {elapsed:.0f}s elapsed")
    del df
    gc.collect()

print(f"\nAll {len(parts)} parts processed in {time.time()-t_proc:.0f}s")
print(f"Unique accounts with transactions: {len(acc.txn_count):,}")

## 4 — Process transactions_additional (Geo, IP, Balance)

In [ ]:
# Geo/IP/Balance features from transactions_additional
add_parts = sorted(glob(f"{DATA_DIR}/transactions_additional/batch-*/part_*.parquet"))
print(f"Processing {len(add_parts)} additional transaction parts...")
t_add = time.time()

# Accumulators for additional features
unique_ips = defaultdict(set)        # account_id -> set of IPs
lat_lons = defaultdict(list)         # account_id -> list of (lat, lon)
balance_list = defaultdict(list)     # account_id -> list of balance_after_transaction
sub_type_counter = defaultdict(Counter)  # account_id -> Counter of transaction_sub_type
part_type_counter = defaultdict(Counter)  # account_id -> Counter of part_transaction_type

# We need account_id from the main transactions — match by transaction_id
# Strategy: read both in parallel and join
main_parts = sorted(glob(f"{DATA_DIR}/transactions/batch-*/part_*.parquet"))

# Build transaction_id -> account_id mapping chunk by chunk
# This is too large to hold in memory. Instead, read both together batch by batch.
batches_main = sorted(set(os.path.dirname(p) for p in main_parts))
batches_add  = sorted(set(os.path.dirname(p) for p in add_parts))

# For each batch, load all main txn parts to get txn_id->account_id, then join with additional
for batch_idx, batch_main_dir in enumerate(batches_main):
    batch_name = os.path.basename(batch_main_dir)
    batch_add_dir = os.path.join(DATA_DIR, "transactions_additional", batch_name)

    if not os.path.isdir(batch_add_dir):
        continue

    # Load all main parts for this batch (just id + account_id)
    main_batch_parts = sorted(glob(os.path.join(batch_main_dir, "part_*.parquet")))
    id_map = pd.concat([
        pd.read_parquet(p, columns=["transaction_id", "account_id"]) for p in main_batch_parts
    ], ignore_index=True)
    id_lookup = dict(zip(id_map["transaction_id"], id_map["account_id"]))

    # Process additional parts for this batch
    add_batch_parts = sorted(glob(os.path.join(batch_add_dir, "part_*.parquet")))
    for ap in add_batch_parts:
        adf = pd.read_parquet(ap)
        adf["account_id"] = adf["transaction_id"].map(id_lookup)
        adf = adf[adf["account_id"].notna()]

        # IP features
        if "ip_address" in adf.columns:
            valid_ip = adf[adf["ip_address"].notna()]
            for aid, ip in zip(valid_ip["account_id"], valid_ip["ip_address"]):
                if len(unique_ips[aid]) < 1000:  # cap
                    unique_ips[aid].add(ip)

        # Geo features
        if "latitude" in adf.columns and "longitude" in adf.columns:
            valid_geo = adf[adf["latitude"].notna() & adf["longitude"].notna()]
            for aid, lat, lon in zip(valid_geo["account_id"], valid_geo["latitude"], valid_geo["longitude"]):
                if len(lat_lons[aid]) < 2000:
                    lat_lons[aid].append((lat, lon))

        # Balance after transaction
        if "balance_after_transaction" in adf.columns:
            valid_bal = adf[adf["balance_after_transaction"].notna()]
            for aid, bal in zip(valid_bal["account_id"], valid_bal["balance_after_transaction"]):
                if len(balance_list[aid]) < 5000:
                    balance_list[aid].append(bal)

        # Sub-type
        if "transaction_sub_type" in adf.columns:
            for (aid, st), cnt in adf[adf["transaction_sub_type"].notna()].groupby(
                ["account_id", "transaction_sub_type"]).size().items():
                sub_type_counter[aid][st] += cnt

        # Part type
        if "part_transaction_type" in adf.columns:
            for (aid, pt), cnt in adf[adf["part_transaction_type"].notna()].groupby(
                ["account_id", "part_transaction_type"]).size().items():
                part_type_counter[aid][pt] += cnt

        del adf
    del id_map, id_lookup
    gc.collect()

    elapsed = time.time() - t_add
    print(f"  Batch {batch_idx+1}/{len(batches_main)} ({batch_name}) done — {elapsed:.0f}s elapsed")

print(f"\nAdditional features processed in {time.time()-t_add:.0f}s")
print(f"Accounts with IP data: {len(unique_ips):,}")
print(f"Accounts with geo data: {len(lat_lons):,}")
print(f"Accounts with balance data: {len(balance_list):,}")

## 5 — Finalize Transaction Features

In [ ]:
print("Finalizing transaction features...")
txn_features = acc.finalize(account_open, MULE_TRIGRAMS_TOP10)
print(f"Transaction features: {txn_features.shape}")

## 6 — Build Additional Features (Geo, IP, Balance, Demographics)

In [ ]:
print("Building additional features from new P2 tables...")

additional_rows = []
for aid in txn_features["account_id"]:
    row = {"account_id": aid}

    # IP features
    ips = unique_ips.get(aid, set())
    row["unique_ip_count"] = len(ips)

    # Geo features
    coords = lat_lons.get(aid, [])
    if len(coords) >= 2:
        lats = [c[0] for c in coords]
        lons = [c[1] for c in coords]
        row["geo_spread_lat"] = max(lats) - min(lats)
        row["geo_spread_lon"] = max(lons) - min(lons)
        row["geo_unique_locations"] = len(set((round(la, 3), round(lo, 3)) for la, lo in coords))
    else:
        row["geo_spread_lat"] = np.nan
        row["geo_spread_lon"] = np.nan
        row["geo_unique_locations"] = 0

    # Balance dynamics
    bals = balance_list.get(aid, [])
    if len(bals) >= 3:
        bal_arr = np.array(bals)
        row["balance_std"] = bal_arr.std()
        row["balance_min"] = bal_arr.min()
        row["zero_crossings"] = int(np.sum(np.diff(np.sign(bal_arr)) != 0))
    else:
        row["balance_std"] = np.nan
        row["balance_min"] = np.nan
        row["zero_crossings"] = 0

    # Transaction sub-type
    st = sub_type_counter.get(aid, Counter())
    total_st = sum(st.values()) or 1
    row["cash_txn_pct"] = st.get("CLT_CASH", 0) / total_st
    row["loan_txn_pct"] = st.get("LOAN", 0) / total_st

    # Part transaction type
    pt = part_type_counter.get(aid, Counter())
    total_pt = sum(pt.values()) or 1
    row["customer_induced_pct"] = pt.get("CI", 0) / total_pt
    row["bank_induced_pct"] = pt.get("BI", 0) / total_pt

    additional_rows.append(row)

f_additional = pd.DataFrame(additional_rows)
print(f"Additional features: {f_additional.shape}")

In [ ]:
# Scheme code (accounts-additional)
scheme_dummies = pd.get_dummies(acct_extra["scheme_code"], prefix="scheme").astype(int)
scheme_dummies["account_id"] = acct_extra["account_id"]
# Flag PMJDY specifically (known mule vector)
scheme_dummies["is_pmjdy"] = (acct_extra["scheme_code"] == "PMJDY").astype(int)
print(f"Scheme features: {scheme_dummies.shape}")

In [ ]:
# Demographics
demo_feats = demographics.merge(linkage[["customer_id", "account_id"]], on="customer_id", how="inner")
demo_feats["is_joint_account"] = (demo_feats["joint_account_flag"] == "Y").astype(int)
demo_feats["is_nri"] = (demo_feats["nri_flag"] == "Y").astype(int)
demo_feats["is_male"] = (demo_feats["gender"] == "M").astype(int)
# Phone sharing — count how many accounts share same phone number
phone_counts = demo_feats.groupby("phone_number")["account_id"].transform("count")
demo_feats["shared_phone_count"] = phone_counts
# Address sharing
addr_counts = demo_feats.groupby("address")["account_id"].transform("count")
demo_feats["shared_address_count"] = addr_counts

demo_feats = demo_feats[["account_id", "is_joint_account", "is_nri", "is_male",
                          "shared_phone_count", "shared_address_count"]]
print(f"Demographics features: {demo_feats.shape}")

In [ ]:
# Branch enrichment
branch_feats = accounts[["account_id", "branch_code"]].merge(
    branch_info, on=None, left_on="branch_code", right_on="branch_code", how="left"
)
branch_feats["is_urban"] = (branch_feats["branch_type"] == "urban").astype(int)
branch_feats["is_rural"] = (branch_feats["branch_type"] == "rural").astype(int)
for col in ["branch_employee_count", "branch_turnover", "branch_asset_size"]:
    if col in branch_feats.columns:
        branch_feats[col] = pd.to_numeric(branch_feats[col], errors="coerce")
branch_feats = branch_feats[["account_id", "is_urban", "is_rural",
                              "branch_employee_count", "branch_turnover", "branch_asset_size"]]
print(f"Branch features: {branch_feats.shape}")

## 7 — Static Features (Metadata, Freeze, Branch Encoding)

In [ ]:
# H9 — Freeze history
meta = accounts[["account_id", "freeze_date", "unfreeze_date",
                   "account_status", "daily_avg_balance"]].copy()
meta["has_prior_freeze"] = meta["freeze_date"].notna().astype(int)
meta["has_unfreeze"]     = meta["unfreeze_date"].notna().astype(int)
meta["is_frozen"]        = (meta["account_status"] == "frozen").astype(int)

# H38 — Residual ratio
meta = meta.merge(
    txn_features[["account_id", "total_volume"]], on="account_id", how="left"
)
meta["residual_ratio"] = meta["daily_avg_balance"].abs() / meta["total_volume"].clip(lower=1)
meta["is_tiny_residual"] = (meta["residual_ratio"] < 0.001).astype(int)

# R8 — Mobile change spike (simplified: flag if mobile was updated)
meta = meta.merge(
    accounts[["account_id", "last_mobile_update_date"]], on="account_id", how="left"
)
meta["has_mobile_update"] = meta["last_mobile_update_date"].notna().astype(int)

# Aadhaar missing
cust_feats = linkage[["account_id", "customer_id"]].merge(
    customers[["customer_id", "aadhaar_available"]], on="customer_id", how="left"
)
cust_feats["aadhaar_missing"] = (cust_feats["aadhaar_available"].isna() | 
                                   (cust_feats["aadhaar_available"] == "N")).astype(int)

meta = meta.merge(cust_feats[["account_id", "aadhaar_missing"]], on="account_id", how="left")

meta_final = meta[["account_id", "has_prior_freeze", "has_unfreeze", "is_frozen",
                     "residual_ratio", "is_tiny_residual", "has_mobile_update", "aadhaar_missing"]]
print(f"Metadata features: {meta_final.shape}")

In [ ]:
# R12+H13 — Branch mule rate (target-encoded, smoothed)
acct_branch = accounts[["account_id", "branch_code"]].merge(
    labels[["account_id", "is_mule"]], on="account_id", how="inner"
)
global_rate = acct_branch["is_mule"].mean()
bs = acct_branch.groupby("branch_code").agg(n=("is_mule", "count"), pos=("is_mule", "sum")).reset_index()
alpha = 10
bs["branch_mule_rate"] = (bs["pos"] + alpha * global_rate) / (bs["n"] + alpha)
branch_rate = accounts[["account_id", "branch_code"]].merge(
    bs[["branch_code", "branch_mule_rate"]], on="branch_code", how="left"
)
branch_rate["branch_mule_rate"] = branch_rate["branch_mule_rate"].fillna(global_rate)
branch_rate = branch_rate[["account_id", "branch_mule_rate"]]
print(f"Branch encoding: {branch_rate.shape}")

In [ ]:
# H10 — MCC concentration (top-5 mule-enriched MCCs from training data)
mcc_mule = txn_features.merge(labels[["account_id", "is_mule"]], on="account_id", how="inner")
# We already have mcc_entropy; for MCC concentration we'd need per-MCC mule rates
# Use the mcc_entropy as a proxy (already computed in txn_features)
print("MCC entropy already computed in txn_features (mcc_entropy)")

## 8 — Merge All Features

In [ ]:
print("Merging all feature groups...")
features = all_ids.copy()

for name, df in [
    ("TxnFeatures",   txn_features),
    ("Additional",    f_additional),
    ("Scheme",        scheme_dummies),
    ("Demographics",  demo_feats),
    ("BranchInfo",    branch_feats),
    ("Metadata",      meta_final),
    ("BranchRate",    branch_rate),
]:
    before = features.shape[1]
    features = features.merge(df, on="account_id", how="left")
    print(f"  {name:<15} → +{features.shape[1]-before} cols, total={features.shape[1]}")

print(f"\nMerged shape: {features.shape}")

## 9 — Interaction Features (H42) & Composite Score (R10)

In [ ]:
# H42 — Interaction terms
features["degree_x_night"]   = features["degree_centrality"] * features["night_txn_pct"]
features["gapcv_x_degree"]   = features["gap_cv"] * features["degree_centrality"]
features["gapcv_x_near50k"]  = features["gap_cv"] * features["near_threshold_pct"]
features["degree_x_near50k"] = features["degree_centrality"] * features["near_threshold_pct"]

# R10 — Composite score
score_cols = [c for c in [
    "near_threshold_pct", "round_amount_pct", "gap_cv", "degree_centrality",
    "mule_trigram_count", "branch_mule_rate", "has_prior_freeze", "mcc_entropy"
] if c in features.columns]
for col in score_cols:
    m = features[col].mean()
    s = features[col].std()
    features[col + "_z"] = (features[col] - m) / s if s > 0 else 0
features["composite_score"] = features[[c + "_z" for c in score_cols]].sum(axis=1)
# Drop intermediate z-score columns
features.drop(columns=[c + "_z" for c in score_cols], inplace=True)

print(f"Added interactions + composite → total: {features.shape[1]} columns")

## 10 — Quality Check

In [ ]:
nan_pct = features.drop(columns=["account_id"]).isna().mean().sort_values(ascending=False)
print("Features with >50% NaN:")
high = nan_pct[nan_pct > 0.5]
if len(high):
    print(high.to_string())
else:
    print("  None!")
print(f"\nTotal features: {features.shape[1] - 1}")
print(f"Features with 0% NaN: {(nan_pct == 0).sum()}")

In [ ]:
# Correlation with target
train_f = features.merge(labels[["account_id", "is_mule"]], on="account_id", how="inner")
print(f"Train shape: {train_f.shape}")
# Only numeric columns
num_cols = train_f.select_dtypes(include=[np.number]).columns.tolist()
if "is_mule" in num_cols:
    corr = train_f[num_cols].corr()["is_mule"].drop("is_mule", errors="ignore")
    corr = corr.abs().sort_values(ascending=False)
    print(f"\nTop 20 features by |correlation| with is_mule:")
    print(corr.head(20).to_string())

## 11 — Save Features

In [ ]:
train_out = features[features["account_id"].isin(labels["account_id"])].copy()
test_out  = features[features["account_id"].isin(test["account_id"])].copy()
train_out = train_out.merge(labels[["account_id", "is_mule"]], on="account_id", how="left")

print(f"Train: {train_out.shape} (mule rate: {train_out['is_mule'].mean():.4f})")
print(f"Test:  {test_out.shape}")

train_out.to_csv("features_train_p2.csv", index=False)
test_out.to_csv("features_test_p2.csv", index=False)

print(f"\n✅ Saved features_train_p2.csv")
print(f"✅ Saved features_test_p2.csv")
print(f"Total time: {time.time()-t0:.0f}s")

In [ ]:
feature_cols = [c for c in features.columns if c not in ["account_id"]]
print(f"\n{'='*60}")
print(f"FEATURE LIST ({len(feature_cols)} features)")
print(f"{'='*60}")
for i, col in enumerate(feature_cols, 1):
    nan_rate = features[col].isna().mean()
    print(f"  {i:2d}. {col:<35} NaN={nan_rate:.1%}")